# Lab 03 Extra: Multi-Agent LLM Mafia

This is a companion demo to `Lab_03_Knowledge.ipynb`, not a replacement for it.

In the main Lab 03 notebook, we represented knowledge with **propositional logic** and used a **model checker** (`sympy.logic`) to prove facts in the Cluedo game — every rule was hand-encoded, and the machinery guaranteed a correct answer.

Here, six separate LLM instances play a full game of **Mafia** against each other. Nobody hand-codes the inference rules this time. Each AI agent has to track *who knows what*, notice contradictions, and reason about hidden state using nothing but natural language — the same underlying problem as Cluedo, but solved (and exploited) very differently.

Watch for:
- **Knowledge inference** — agents inferring hidden roles from public statements and voting patterns.
- **Deception** — the Mafia agent is explicitly told it may lie. Watch its private reasoning vs. its public statement.
- **Multi-agent collaboration (and its failure modes)** — Village-aligned agents must coordinate with no shared ground truth, and sometimes get it badly wrong.
- **Model intelligence, side by side** — this game's roster deliberately mixes 3 flagship-tier models with 3 budget-tier models. Watch for differences in how convincingly each argues, lies, or gets caught.

## Why this is a replay, not a live demo

This notebook **does not call any LLM API**. Every game shown here was generated ahead of time by `generate_games.py` and saved as a JSON file in `games/`. This notebook only reads and displays that file.

Why: live API calls in front of a class risk rate limits, timeouts, unpredictable pacing, and real cost every time the notebook is re-run. Pre-generating games lets us pick good ones ahead of time and replay them instantly and for free. See `DESIGN.md` in this folder for the full reasoning.

**Note:** the game picker below uses an interactive dropdown (`ipywidgets`), so it needs to be run live in Jupyter or Colab — it will show as an empty placeholder in a statically-exported HTML/PDF version of this notebook, or in a screenshot taken before you've selected a game. That's expected; just run the cells live.

If you have your own OpenRouter API key, you can generate new games yourself — see the last section of this notebook.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import replay_utils as ru

## The roster

Every game uses the same fixed 6-model roster: 3 flagship-tier models and 3 budget-tier models, spanning 4 vendors. Which *role* (Mafia / Detective / Doctor / Villager) each model plays is randomized per game — so across multiple games you can compare how the *same* model behaves under different roles, and how models of different tiers stack up against each other.

| Tier | Vendor | Model |
|---|---|---|
| Flagship | OpenAI | `openai/gpt-6-astra` |
| Flagship | Anthropic | `anthropic/claude-sonnet-5` |
| Flagship | Anthropic | `anthropic/claude-opus-5` |
| Budget | Google | `google/gemini-3.8-flash` |
| Budget | DeepSeek | `deepseek/deepseek-v4-flash-0731` |
| Budget | OpenAI | `openai/gpt-5.4-mini` |

**Rules (deliberately simple):** 6 players — 1 Mafia, 1 Detective, 1 Doctor, 3 Villagers. Each round: night actions (Doctor protects, Mafia kills, Detective investigates) → morning death announcement → one public statement per living player → a vote to eliminate someone. Mafia wins once Mafia count ≥ remaining Villagers; Village wins once all Mafia are eliminated.

**Roles are hidden throughout the game** and only revealed together at the very end — so a player who gets voted out mid-game isn't necessarily confirmed as the Mafia. Try guessing as you read!

## Play back a game, step by step

Pick a game from the dropdown, then click **Next ▶** to advance one stage at a time: night actions → morning announcement → day discussion → vote, repeating each round, ending with the reveal. Each click shows only what's new — the player cards at the top update in place (alive/eliminated) as you go, instead of dumping the entire game at once.

Click the **"Roles hidden — click to reveal roles"** label above the player cards at any time to toggle whether roles are shown — independent of how far you've stepped through the game, so you (or a curious student) can peek without spoiling the pacing for everyone else watching.

In [2]:
step_browser = ru.GameBrowser("games", mode="step")
step_browser.show()

Dropdown(description='Game:', layout=Layout(width='600px'), options=(('mafia_2026-09-09_035509  —  3 round(s),…

Output()

## Optional: see an entire game at once

If you'd rather skim a full game in one scroll (e.g. reviewing a game on your own after class) instead of clicking through it stage by stage, pick a game below — this renders every round's night/morning/discussion/vote in one long stack, ending in the reveal block.

In [3]:
browser = ru.GameBrowser("games", mode="full")
browser.show()

Dropdown(description='Game:', layout=Layout(width='600px'), options=(('mafia_2026-09-09_035509  —  3 round(s),…

Output()

---
## (Optional, for students with their own OpenRouter key) Generate a new game

This notebook is a **replay only** — it never asks for or uses an API key. If you want to generate a fresh game yourself instead of only replaying pre-made ones, see **`SETUP.md`** in this folder for the full steps.

**Read the API key safety section in `SETUP.md` before doing this** — an API key is tied to your own billing, and it's easy to accidentally leak one by pasting it somewhere you didn't mean to (a notebook cell, a screenshot, a homework submission). `SETUP.md` covers what to avoid and how to set a spending limit.

Once you've generated a new game, its file will appear in `games/` — re-run the dropdown cells above and it'll show up as a new option automatically.

**Discussion prompt (optional):** after generating your own game, open the JSON file directly and compare each agent's `private_reasoning` to their `public_statement` in the day-discussion entries. Where did an agent say something different from what it was actually thinking? Was it lying, bluffing, or just being diplomatic?

---
### HW3-5: Multi-Agent Mafia

Pick **1–2 games** from the dropdown above and watch them closely (step-by-step mode is recommended so you can pause and think between stages). For each game, note the **game filename** (e.g. `mafia_2026-09-09_18fca6.json`) so your answers are traceable to a specific recorded game — someone should be able to load your exact game and see what you saw.

Answer the following using # comments or insert a markdown cell.

#### HW3-5-1

Which game(s) did you watch? List the game filename(s) here.

In [4]:
# Your answer here

#### HW3-5-2

Before revealing the roles, write down your own guess: who did you think was the Mafia, and why? Quote or paraphrase the specific statement(s) that made you suspicious.

In [5]:
# Your answer here

#### HW3-5-3

Now reveal the roles. Were you right? If you were wrong, what threw you off — was the real Mafia unusually convincing, or did an innocent player just look suspicious by chance?

In [6]:
# Your answer here

#### HW3-5-4

Which agent do you think played its role the best — regardless of which team it was on? What specifically did it do (a clever lie, a well-timed claim, a convincing accusation, spotting someone else's contradiction) that made it effective?

In [7]:
# Your answer here

#### HW3-5-5

The roster mixes flagship-tier models (`gpt-6-astra`, `claude-sonnet-5`, `claude-opus-5`) with budget-tier models (`gemini-3.8-flash`, `deepseek-v4-flash-0731`, `gpt-5.4-mini`). Based on what you watched, which model seemed the *most capable* — best reasoning, hardest to catch in a contradiction, most persuasive? Which seemed the *weakest*? Was the result what you expected from a flagship-vs-budget matchup, or did a cheaper model outperform a more expensive one anywhere?

In [8]:
# Your answer here

#### HW3-5-6

Open the game's JSON file directly and find one moment where an agent's `private_reasoning` says something noticeably different from its `public_statement` in the same turn. Quote both. What does the gap tell you the agent was doing — managing its image, protecting a teammate, testing a bluff, something else?

In [9]:
# Your answer here

#### HW3-5-7

Overall, did these AI agents feel human to you while you were watching — or did something consistently feel "off," artificial, or too mechanical? Give at least one concrete example of a moment that felt convincingly human, and one that felt distinctly not human.

In [10]:
# Your answer here

#### HW3-5-8

Connect this back to the Cluedo/propositional-logic section of this lab. There, a model checker gave you a *provably correct* answer once the rules and clues were encoded. Here, no one hand-coded the inference rules — the agents reasoned in natural language and could be wrong, inconsistent, or deceived. In your own words, what is the tradeoff between these two approaches to "knowledge and reasoning"? When would you want the guaranteed-correct symbolic approach, and when might a flexible-but-fallible LLM agent be more useful?

In [11]:
# Your answer here